Make sure the right schema is used

In [0]:
USE CATALOG sac;

USE SCHEMA customer_service;

# Create silver tables
customer

In [0]:
CREATE TABLE IF NOT EXISTS customer (
        customer_id STRING NOT NULL,
        signup_date DATE,
        plan_tier STRING,
        address STRING,
        zone STRING,
        contract_type STRING,
        autopay_enabled BOOLEAN,
        account_age_months DOUBLE,
        monthly_bill DOUBLE,
        speed_tier_mbps INT,
        data_usage_gb_last_month DOUBLE,
        CONSTRAINT customer_pk PRIMARY KEY (customer_id)
    );

ALTER TABLE customer DROP CONSTRAINT IF EXISTS customer_sd;
ALTER TABLE customer DROP CONSTRAINT IF EXISTS customer_aa;
ALTER TABLE customer DROP CONSTRAINT IF EXISTS customer_mb;
ALTER TABLE customer DROP CONSTRAINT IF EXISTS customer_st;
ALTER TABLE customer DROP CONSTRAINT IF EXISTS customer_du;

ALTER TABLE customer ADD CONSTRAINT customer_sd CHECK (signup_date <= CURRENT_DATE());

ALTER TABLE customer ADD CONSTRAINT customer_aa CHECK (account_age_months >= 0);

ALTER TABLE customer ADD CONSTRAINT customer_mb CHECK (monthly_bill >= 0);

ALTER TABLE customer ADD CONSTRAINT customer_st CHECK (speed_tier_mbps >= 0);

ALTER TABLE customer ADD CONSTRAINT customer_du CHECK (data_usage_gb_last_month >= 0);

churn

In [0]:
CREATE TABLE IF NOT EXISTS churn (
        customer_id STRING NOT NULL,
        churned BOOLEAN,
        churn_date DATE,
        churn_reason STRING,
        CONSTRAINT churn_pk PRIMARY KEY (customer_id),
        CONSTRAINT churn_fk FOREIGN KEY (customer_id) REFERENCES customer (customer_id)
    );

ALTER TABLE churn DROP CONSTRAINT IF EXISTS churn_cd;

ALTER TABLE churn ADD CONSTRAINT churn_cd CHECK (churn_date IS NULL OR churn_date <= CURRENT_DATE());

connection logs

In [0]:
CREATE TABLE IF NOT EXISTS log (
        log_id STRING NOT NULL,
        customer_id STRING,
        timestamp DATE,
        speed_measured_mbps DOUBLE,
        packet_loss_percent DOUBLE,
        latency_ms DOUBLE,
        downtime_minutes INTEGER,
        connection_drops_count INTEGER,
        issue_detected STRING,
        CONSTRAINT log_pk PRIMARY KEY (log_id),
        CONSTRAINT log_fk FOREIGN KEY (customer_id) REFERENCES customer (customer_id)
    );

ALTER TABLE log DROP CONSTRAINT IF EXISTS log_connection_ts;
ALTER TABLE log DROP CONSTRAINT IF EXISTS log_connection_sm;
ALTER TABLE log DROP CONSTRAINT IF EXISTS log_connection_pl;
ALTER TABLE log DROP CONSTRAINT IF EXISTS log_connection_l;
ALTER TABLE log DROP CONSTRAINT IF EXISTS log_connection_dt;
ALTER TABLE log DROP CONSTRAINT IF EXISTS log_connection_cd;

ALTER TABLE log ADD CONSTRAINT log_connection_ts CHECK(timestamp <= CURRENT_DATE());

ALTER TABLE log ADD CONSTRAINT log_connection_sm CHECK(speed_measured_mbps >= 0);
        
ALTER TABLE log ADD CONSTRAINT log_connection_pl CHECK(
        packet_loss_percent >= 0
        AND packet_loss_percent <= 100
    );

ALTER TABLE log ADD CONSTRAINT log_connection_l CHECK(latency_ms >= 0);

ALTER TABLE log ADD CONSTRAINT log_connection_dt CHECK(downtime_minutes >= 0);

ALTER TABLE log ADD CONSTRAINT log_connection_cd CHECK(connection_drops_count >= 0);

support ticket

In [0]:
CREATE TABLE IF NOT EXISTS ticket (
        ticket_id STRING NOT NULL,
        customer_id STRING,
        timestamp_created TIMESTAMP,
        timestamp_closed TIMESTAMP,
        subject STRING,
        description STRING,
        category STRING,
        priority STRING,
        channel STRING,
        status STRING,
        solved_in_hours INT,
        log_id STRING,
        technical_issue_type STRING,
        CONSTRAINT ticket_pk PRIMARY KEY (ticket_id),
        CONSTRAINT ticket_f1 FOREIGN KEY (customer_id) REFERENCES customer (customer_id),
        CONSTRAINT ticket_f2 FOREIGN KEY (log_id) REFERENCES log (log_id)
    );

ALTER TABLE ticket DROP CONSTRAINT IF EXISTS support_ticket_ci;
ALTER TABLE ticket DROP CONSTRAINT IF EXISTS support_ticket_tc;
ALTER TABLE ticket DROP CONSTRAINT IF EXISTS support_ticket_tl;
ALTER TABLE ticket DROP CONSTRAINT IF EXISTS support_ticket_sh;

ALTER TABLE ticket ADD CONSTRAINT support_ticket_ci CHECK(customer_id IS NOT NULL);

ALTER TABLE ticket ADD CONSTRAINT support_ticket_tc CHECK(
        timestamp_created <= timestamp_closed
    );

ALTER TABLE ticket ADD CONSTRAINT support_ticket_tl CHECK(
        timestamp_created <= CURRENT_DATE()
        OR timestamp_closed <= CURRENT_DATE()
    );

ALTER TABLE ticket ADD CONSTRAINT support_ticket_sh CHECK(solved_in_hours >= 0);

agent

In [0]:
CREATE TABLE IF NOT EXISTS agent (
        agent_id STRING NOT NULL,
        first_name STRING,
        last_name STRING,
        employment_date DATE,
        experience_level STRING,
        employment_months DOUBLE,
        monthly_salary_eur INTEGER,
        CONSTRAINT agent_pk PRIMARY KEY (agent_id)
    );

ALTER TABLE agent DROP CONSTRAINT IF EXISTS agent_ed;
ALTER TABLE agent DROP CONSTRAINT IF EXISTS agent_em;
ALTER TABLE agent DROP CONSTRAINT IF EXISTS agent_ms;

ALTER TABLE agent ADD CONSTRAINT agent_ed CHECK(employment_date <= CURRENT_DATE());

ALTER TABLE agent ADD CONSTRAINT agent_em CHECK(employment_months >= 0);

ALTER TABLE agent ADD CONSTRAINT agent_ms CHECK(monthly_salary_eur >= 0);

chat

In [0]:

CREATE TABLE IF NOT EXISTS chat (
        session_id STRING NOT NULL,
        customer_id STRING,
        agent_id STRING,
        timestamp_start TIMESTAMP,
        timestamp_end TIMESTAMP,
        CONSTRAINT chat_pk PRIMARY KEY (session_id),
        CONSTRAINT chat_f1 FOREIGN KEY (customer_id) REFERENCES customer (customer_id),
        CONSTRAINT chat_f2 FOREIGN KEY (agent_id) REFERENCES agent (agent_id)
    );

ALTER TABLE chat DROP CONSTRAINT IF EXISTS chat_transcript_ci;
ALTER TABLE chat DROP CONSTRAINT IF EXISTS chat_transcript_ai;
ALTER TABLE chat DROP CONSTRAINT IF EXISTS chat_transcript_tc;
ALTER TABLE chat DROP CONSTRAINT IF EXISTS chat_transcript_te;

ALTER TABLE chat ADD CONSTRAINT chat_transcript_ci CHECK(customer_id IS NOT NULL);

ALTER TABLE chat ADD CONSTRAINT chat_transcript_ai CHECK(agent_id IS NOT NULL);

ALTER TABLE chat ADD CONSTRAINT chat_transcript_tc CHECK(
        timestamp_start <= timestamp_end
    );

ALTER TABLE chat ADD CONSTRAINT chat_transcript_te CHECK(
        timestamp_start <= CURRENT_DATE()
        OR timestamp_end <= CURRENT_DATE()
    );

message

In [0]:
CREATE TABLE IF NOT EXISTS message (
        session_id STRING NOT NULL,
        speaker STRING,
        timestamp TIMESTAMP NOT NULL,
        message STRING,
        classification STRING,
        comment STRING,
        sentiment STRING,
        CONSTRAINT message_fk FOREIGN KEY (session_id) REFERENCES chat (session_id),
        CONSTRAINT message_pk PRIMARY KEY (session_id, speaker, timestamp)
    );

ALTER TABLE message DROP CONSTRAINT IF EXISTS message_ts;

ALTER TABLE message ADD CONSTRAINT message_ts CHECK(timestamp IS NOT NULL AND timestamp <= CURRENT_TIMESTAMP());

# Fill tables with content
customer

In [0]:
WITH location_zone AS (
    SELECT
        customer_id,
        ai_query(
            'databricks-meta-llama-3-3-70b-instruct',
            "Can you tell me the name of the German federal state (Bundesland) that serves the provided address? Please just name the state with no additional text. The address: "
                || address
        ) AS location_zone
    FROM
        customer_bronze
) MERGE INTO
    customer s
USING (
    SELECT
        c.customer_id,
        CAST(c.signup_date AS DATE) AS signup_date,
        c.plan_tier,
        c.address,
        regexp_extract(c.address, '(/d{5})') AS zip,
        CASE
            WHEN LOWER(l.location_zone) IN ('lower saxony', 'niedersachsen') THEN 'Niedersachsen'
            WHEN LOWER(l.location_zone) IN ('saxony-anhalt', 'sachsen-anhalt') THEN 'Sachsen-Anhalt'
            WHEN LOWER(l.location_zone) IN ('saxony', 'sachsen') THEN 'Sachsen'
            WHEN
                LOWER(l.location_zone) IN (
                    'mecklenburg-western pomerania', 'mecklenburg-vorpommern'
                )
            THEN
                'Mecklenburg-Vorpommern'
            WHEN LOWER(l.location_zone) IN ('brandenburg') THEN 'Brandenburg'
            WHEN LOWER(l.location_zone) IN ('baden-württemberg') THEN 'Baden-Württemberg'
            WHEN LOWER(l.location_zone) IN ('hesse', 'hessen') THEN 'Hessen'
            WHEN
                LOWER(l.location_zone) IN ('rhineland-palatinate', 'rheinland-pfalz')
            THEN
                'Rheinland-Pfalz'
            WHEN LOWER(l.location_zone) IN ('schleswig-holstein') THEN 'Schleswig-Holstein'
            WHEN
                LOWER(l.location_zone) IN ('north rhine-westphalia', 'nordrhein-westfalen')
            THEN
                'Nordrhein-Westfahlen'
            WHEN LOWER(l.location_zone) IN ('saarland') THEN 'Saarland'
            WHEN LOWER(l.location_zone) IN ('bremen') THEN 'Bremen'
            WHEN LOWER(l.location_zone) IN ('hamburg') THEN 'Hamburg'
            WHEN LOWER(l.location_zone) IN ('bavaria', 'bayern') THEN 'Bayern'
            WHEN LOWER(l.location_zone) IN ('thuringia', 'thüringen') THEN 'Thüringen'
            ELSE 'Unbekannt'
        END AS zone,
        c.contract_type,
        CASE
            WHEN LOWER(c.autopay_enabled) LIKE 'true' THEN TRUE
            ELSE FALSE
        END AS autopay_enabled,
        CAST(c.account_age_months AS DOUBLE) AS account_age_months,
        CAST(c.monthly_bill AS DOUBLE) AS monthly_bill,
        CAST(c.speed_tier_mbps AS INTEGER) AS speed_tier_mbps,
        CAST(c.data_usage_gb_last_month AS DOUBLE) AS data_usage_gb_last_month
    FROM
        customer_bronze c
            JOIN location_zone l
                ON l.customer_id = c.customer_id
    QUALIFY
        row_number() OVER (PARTITION BY c.customer_id ORDER BY c.ingestion_time DESC) = 1
) b
ON
    b.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

churn

In [0]:
MERGE INTO
    churn s
USING (
    SELECT
        c.customer_id,
        CASE
            WHEN c.churned = 1 THEN TRUE
            ELSE FALSE
        END AS churned,
        CAST(c.churn_date AS date) AS churn_date,
        c.churn_reason
    FROM
        churn_bronze c
    QUALIFY
        row_number() OVER (PARTITION BY c.customer_id ORDER BY c.ingestion_time DESC) = 1
) b
ON
    b.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

log

In [0]:
MERGE INTO
    log s
USING (
    SELECT
        l.log_id,
        l.customer_id,
        CAST(l.timestamp AS date) AS timestamp,
        CAST(l.speed_measured_mbps AS double) AS speed_measured_mbps,
        CAST(l.packet_loss_percent AS double) AS packet_loss_percent,
        CAST(l.latency_ms AS double) AS latency_ms,
        CAST(l.downtime_minutes AS integer) AS downtime_minutes,
        CAST(l.connection_drops_count AS integer) AS connection_drops_count,
        l.issue_detected
    FROM
        log_bronze l
    QUALIFY
        row_number() OVER (PARTITION BY l.log_id ORDER BY l.ingestion_time DESC) = 1
) b
ON
    s.log_id = b.log_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

ticket

In [0]:
MERGE INTO
    ticket s
USING (
    SELECT
        t.ticket_id,
        t.customer_id,
        CAST(t.timestamp_created AS timestamp) AS timestamp_created,
        CAST(t.timestamp_closed AS timestamp) AS timestamp_closed,
        t.subject,
        t.description,
        t.category,
        t.priority,
        t.channel,
        t.status,
        CAST(t.solved_in_hours AS int) AS solved_in_hours,
        t.log_id,
        t.technical_issue_type
    FROM
        ticket_bronze t
    QUALIFY
        row_number() OVER (PARTITION BY t.ticket_id ORDER BY t.ingestion_time DESC) = 1
) b
ON
    s.ticket_id = b.ticket_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

agent

In [0]:
MERGE INTO
    agent s
USING (
    SELECT
        a.agent_id,
        SUBSTRING(a.agent_name, 1, POSITION(' ' IN a.agent_name) - 1) AS first_name,
        SUBSTRING(a.agent_name, POSITION(' ' IN a.agent_name) + 1) AS last_name,
        TO_DATE(a.employment_date) AS employment_date,
        a.experience_level,
        CAST(a.employment_months AS double) AS employment_months,
        CAST(a.monthly_salary_eur AS int) AS monthly_salary_eur
    FROM
        agent_bronze a
    QUALIFY
        row_number() OVER (PARTITION BY a.agent_id ORDER BY a.ingestion_time DESC) = 1
) b
ON
    s.agent_id = b.agent_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

chat

In [0]:
MERGE INTO
    chat s
USING (
    SELECT
        c.session_id,
        c.customer_id,
        c.agent_id,
        CAST(c.timestamp_start AS timestamp) AS timestamp_start,
        CAST(c.timestamp_end AS timestamp) AS timestamp_end
    FROM
        chat_bronze c
    QUALIFY
        row_number() OVER (PARTITION BY c.session_id ORDER BY c.ingestion_time DESC) = 1
) b
ON
    s.session_id = b.session_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

message

In [0]:
%skip
WITH opinions AS(
    SELECT 
        SELECT
        c.session_id,
        msg.speaker,
        CAST(msg.timestamp AS timestamp) AS timestamp,
        msg.message,
        AI_ANALYZE_SENTIMENT(msg.message) AS sentiment
    FROM
        chat_bronze c
        LATERAL VIEW EXPLODE(FROM_JSON(c.messages, 'ARRAY<MAP<STRING,STRING>>')) AS msg
)
MERGE INTO
    message s
USING (
    SELECT
        c.session_id,
        msg.speaker,
        CAST(msg.timestamp AS timestamp) AS timestamp,
        msg.message,
        AI_ANALYZE_SENTIMENT(msg.message) AS sentiment
    FROM
        chat_bronze c
        LATERAL VIEW EXPLODE(FROM_JSON(c.messages, 'ARRAY<MAP<STRING,STRING>>')) AS msg
    QUALIFY
        row_number() OVER (PARTITION BY c.session_id ORDER BY c.ingestion_time DESC) = 1
) b
ON
    b.session_id = s.session_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;


                You are an internet service provider. Given a piece of text, output an array of json results that extracts key user opinions, a classification, and a Positive, Negative, or Neutral sentiment about that subject. Classifications must be one of the following answers: CONNECTION ISSUES, SLOW SPEED, SERVICE, PRICE, OTHER. You cannot hallucinate your own classification category.

In [0]:
WITH messages_raw AS (
    SELECT
        c.session_id,
        msg.speaker,
        CAST(msg.timestamp AS timestamp) AS timestamp,
        msg.message,
        c.ingestion_time
    FROM
        chat_bronze c
        LATERAL VIEW EXPLODE(FROM_JSON(c.messages, 'ARRAY<MAP<STRING,STRING>>')) AS msg
),
opinion AS (
    SELECT
        messages_raw.session_id,
        messages_raw.timestamp,
        ai_query(
            "databricks-meta-llama-3-3-70b-instruct",
            request =>
                concat(
                    "Du bist ein Internet Service Provider. Gebe basierend auf einem Textabschnitt als Ergebnis ein JSON Array aus, das eine Zusammenfassung, eine Klassifikation und ein Positiv, Negativ oder Neutral Sentiment über das Thema enthält. Klassifiziert muss einer der folgenden Anworten sein: CONNECTION ISSUES, SLOW SPEED, SERVICE, PRICE, OTHER. Du kannst keine Klassifikations Kategorie halluzinieren.

Beispiel:

DOCUMENT
Mein Router startet sich alle 15 Minuten von selbst neu. Das ist super nervig. (Gemessene Geschwindigkeit: 180 Mbps, Issue: packet_loss).

RESULT
[
{'Classification': 'CONNECTION ISSUES','Comment': 'Router startet ständig neu','Sentiment': 'Negativ'}
]

DOCUMENT\n",
                    messages_raw.message,
                    '\n\nRESULT\n'
                ),
            responseFormat =>
                '{
                "type": "json_schema",
                "json_schema": {
                    "name": "opinion_mining_schema",
                    "schema": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "classification": { "type": "string" },
                                "comment": { "type": "string" },
                                "sentiment": { "type": "string" }
                            }
                        }
                    },
                    "strict": true
                }
            }'
        ) as extracted_opinions
    FROM
        messages_raw
),
messages AS (
    SELECT
        m.session_id,
        m.speaker,
        m.timestamp,
        m.message,
        op.classification,
        CASE 
            WHEN op.comment IS NULL THEN 'Unbekannt' 
            ELSE op.comment
        END AS comment,
        CASE 
            WHEN op.sentiment IN ('Positive', 'Positiv', 'Positives') THEN 'Positiv'
            WHEN op.sentiment IN ('Neutral', 'Neutrales') THEN 'Neutral'
            WHEN op.sentiment IN ('Negative', 'Negativ', 'Negatives') THEN 'Negativ'
            ELSE 'Unbekannt'
        END AS sentiment,
        m.ingestion_time
    FROM
        messages_raw m
            LEFT JOIN opinion o
                ON m.session_id = o.session_id
                AND m.timestamp = o.timestamp
        LATERAL VIEW OUTER EXPLODE(FROM_JSON(o.extracted_opinions, 'ARRAY<MAP<STRING,STRING>>')) AS op
) MERGE INTO
    message s
USING (
    SELECT
        session_id,
        speaker,
        timestamp,
        message,
        classification,
        comment,
        sentiment
    FROM
        messages
    QUALIFY
        row_number() OVER (PARTITION BY session_id, timestamp ORDER BY ingestion_time DESC)
            = 1
) b
ON
    b.session_id = s.session_id
    AND b.timestamp = s.timestamp
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;


MERGE INTO
    message m
USING (
    SELECT
        session_id,
        speaker,
        timestamp,
        message,
        CAST(NULL AS string) AS classification,
        comment,
        sentiment
    FROM
        message
    WHERE
        classification IS NOT NULL
    QUALIFY row_number() OVER (PARTITION BY session_id ORDER BY timestamp) > 1
) mn
ON
    m.session_id = mn.session_id
    AND m.timestamp = mn.timestamp
WHEN MATCHED THEN UPDATE SET m.classification = mn.classification;